In [ ]:
import importlib.util
if importlib.util.find_spec("spytial") is None:
    try:
        import piplite
        await piplite.install("spytial-diagramming")
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "spytial-diagramming"])
from spytial import *
import spytial.utils as _su
if not _su.is_notebook():
    _su.is_notebook = lambda: True  # JupyterLite/Pyodide: render diagram() inline
from spytial.annotations import *

# Max heap (array-backed): step-by-step

A binary max-heap stored in an array. The cell at index `i` is the
parent of `2i+1` and `2i+2`, so each value's spatial position is pinned
by *which slot it currently lives in*.

Every step of `MAX-HEAPIFY` (and of the sift-up inside `INSERT`) is a
swap: two cells exchange values. That's a forced 2-cycle in the
atom→position mapping — no policy can suppress the motion, because the
algorithm itself is the motion.

So we pick `sequence_policy="change_emphasis"`: the viewer's eye is
sent straight to the two cells that just traded values, which is the
entire semantic content of a heapify step. (Try toggling to `stability`
in the dropdown to feel the difference: under stability the swap still
happens, but you're left tracking *where each value went* instead of
*what just got compared*.)

In [ ]:
from typing import List

HEAP_VALUES = "((int - 0).(list.idx))"
LEFT_CHILDREN = "(~(list.idx)).({ i, j : (int-0) | j = multiply[i,2] }).(list.idx)"
RIGHT_CHILDREN = "(~(list.idx)).({ i, j : (int-0) | j = add[1, multiply[i, 2]] }).(list.idx)"

@hideAtom(selector=f'MaxHeap + list + (int - {HEAP_VALUES})')
@orientation(selector=LEFT_CHILDREN, directions=["left", "below"])
@inferredEdge(selector=LEFT_CHILDREN, name="left")
@orientation(selector=RIGHT_CHILDREN, directions=["right", "below"])
@inferredEdge(selector=RIGHT_CHILDREN, name="right")
class MaxHeap:
    """CLRS-style max heap (1-indexed; a[0] unused) instrumented for sequence recording."""

    def __init__(self, data: List[int] = None, seq=None):
        self.a: List[int] = [0]
        self._seq = seq
        if data:
            self.a.extend(data)
        self.n = len(self.a) - 1
        self._snap("initial array")
        if self.n > 1:
            self.build_max_heap()

    def _snap(self, label):
        if self._seq is not None:
            self._seq.record(self, label=label)

    @staticmethod
    def _parent(i): return i // 2
    @staticmethod
    def _left(i):   return 2 * i
    @staticmethod
    def _right(i):  return 2 * i + 1

    def _max_heapify(self, i):
        while True:
            l, r = self._left(i), self._right(i)
            largest = i
            if l <= self.n and self.a[l] > self.a[largest]:
                largest = l
            if r <= self.n and self.a[r] > self.a[largest]:
                largest = r
            if largest == i:
                break
            self.a[i], self.a[largest] = self.a[largest], self.a[i]
            self._snap(f"MAX-HEAPIFY: swap a[{i}] <-> a[{largest}]")
            i = largest

    def build_max_heap(self):
        for i in range(self.n // 2, 0, -1):
            self._max_heapify(i)

    def extract_max(self):
        if self.n < 1:
            raise IndexError("heap underflow")
        m = self.a[1]
        self.a[1] = self.a[self.n]
        self.a.pop()
        self.n -= 1
        self._snap(f"EXTRACT-MAX: extracted {m}, moved last to root")
        if self.n >= 1:
            self._max_heapify(1)
        return m

    def insert(self, key):
        self.n += 1
        self.a.append(key)
        self._snap(f"INSERT({key}): appended at a[{self.n}]")
        i = self.n
        while i > 1 and self.a[self._parent(i)] < self.a[i]:
            p = self._parent(i)
            self.a[i], self.a[p] = self.a[p], self.a[i]
            self._snap(f"sift-up: swap a[{i}] <-> a[{p}]")
            i = p

    def __repr__(self):
        return f"MaxHeap({self.a[1:]})"

## Demo

Build a heap from CLRS Fig. 6.4’s starting array, then extract the max twice. Every swap is one frame.

In [ ]:
seq = sequence(
    sequence_policy="change_emphasis",
    title="Max heap: BUILD-MAX-HEAP + EXTRACT-MAX (one swap per frame)",
)

# CLRS Fig. 6.4 starting array — classic BUILD-MAX-HEAP example.
h = MaxHeap([4, 1, 3, 2, 16, 9, 10, 14, 8, 7], seq=seq)

# Two extractions to also exercise sift-down from the root.
h.extract_max()
h.extract_max()

seq.diagram(method="inline")